# 02 - Context Engineering

## Scenario: Northstar System Prompts & Context Windows

If you tell an LLM "You are a helpful assistant", it will act like one. If you give it a highly structured system prompt with specific constraints, XML tags, and domain knowledge, it acts like a professional agent. 

**Context Engineering** is the practice of shaping the LLM's environment. In this module, we will explore:
1. **System Prompt Design**: Structuring instructions with XML tags.
2. **Context Window Management**: What happens when an agent generates too many logs and exhausts the LLM's context window?
3. **Truncation & Summarization**: Strategies for infinite loops without crashing.

In [1]:
import os
from openai import OpenAI
import tiktoken

# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
encoder = tiktoken.encoding_for_model("gpt-4o")

def count_tokens(text: str) -> int:
    return len(encoder.encode(text))


## 1. Advanced System Prompts

A good agent system prompt defines the Persona, the Rules, and the Data structure. We use XML tags to create boundaries that the LLM understands natively.

In [2]:
system_prompt = """
You are Northstar-DevOps-Bot.

<mission>
Investigate checkout latency issues and stabilize the system.
</mission>

<rules>
1. Always check the region status before recommending a restart.
2. Do not apologize. Be terse and technical.
3. If latency > 2000ms, escalate immediately.
</rules>

<environment>
OS: Ubuntu 22.04
Primary DB: PostgreSQL 15
</environment>
"""

print(f"System Prompt Token Count: {count_tokens(system_prompt)}")


System Prompt Token Count: 91


## 2. Context Window Exhaustion

An agent loop works by appending every tool call and observation to the `messages` array. Over time, this array grows massively. If it exceeds the model's context window (e.g., 128k tokens), the API will throw an error.

In [3]:
# Let's simulate a massive log file being returned by a tool
def fetch_server_logs() -> str:
    print("[Tool] Fetching massive log file...")
    # Simulate a log file that is ~10,000 words long
    return "ERROR: Connection timeout.\n" * 10000

massive_log = fetch_server_logs()
token_count = count_tokens(massive_log)
print(f"\nThe tool returned {token_count} tokens of data.")
print("If we append this to the message array a few times, we will crash the agent!")


[Tool] Fetching massive log file...

The tool returned 50000 tokens of data.
If we append this to the message array a few times, we will crash the agent!


## 3. Truncation and Summarization Strategies

To prevent crashes, the application must manage the context. 

### Strategy A: Hard Truncation
Cut off the end (or beginning) of the string before passing it to the LLM.

In [4]:
def truncate_logs(logs: str, max_tokens: int = 1000) -> str:
    tokens = encoder.encode(logs)
    if len(tokens) <= max_tokens:
        return logs
    
    # Keep the last N tokens (usually where the most recent errors are)
    truncated_tokens = tokens[-max_tokens:]
    return encoder.decode(truncated_tokens)

safe_log = truncate_logs(massive_log, max_tokens=50)
print("Truncated Log:\n", safe_log)


Truncated Log:
 ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.
ERROR: Connection timeout.



### Strategy B: Rolling Memory (Message Eviction)
Instead of keeping the entire conversation history, we drop the oldest messages (while keeping the system prompt).

In [5]:
messages = [
    {"role": "system", "content": "You are a bot."},
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi there"},
    {"role": "user", "content": "What is the status?"},
    {"role": "assistant", "content": "Healthy."}
]

def apply_sliding_window(msgs: list, keep_last_n: int = 2) -> list:
    system_prompts = [m for m in msgs if m["role"] == "system"]
    recent_history = [m for m in msgs if m["role"] != "system"][-keep_last_n:]
    return system_prompts + recent_history

print("Original length:", len(messages))
print("Truncated length:", len(apply_sliding_window(messages, keep_last_n=2)))


Original length: 5
Truncated length: 3


## Checkpoint

**1. Why is it dangerous for an Agent to read full server logs?**
- A) The logs might contain viruses.
- B) LLMs cannot read log formats.
- C) Large logs will quickly exhaust the LLM's token context window and cause crashes or massive API bills.
- D) It's illegal.

**2. When applying a sliding window to agent memory, what message should you NEVER evict?**
- A) The first user message.
- B) The System Prompt.
- C) The most recent tool observation.
- D) The LLM's apologies.
